# Ceph RGW — IAM / STS / S3 Tenant Demo

This notebook demonstrates a complete multi-tenant identity and access management workflow
using the **Ceph RADOS Gateway (RGW)** IAM-compatible API.

| Step | What happens |
|------|--------------|
| **1** | Install libs and configure admin client |
| **2** | Platform admin creates a **tenant** with a **Tenant Administrator** user |
| **3** | Tenant Admin creates **subaccount** users within the tenant |
| **4** | Tenant Admin creates a **tenant-scoped IAM role** with an S3 permission policy |
| **5** | Spin up a local **OIDC / JWKS server** and register it with RGW |
| **6** | Generate a signed **ID token** (JWT) for the subaccount user |
| **7** | Subaccount calls **`AssumeRoleWithWebIdentity`** to obtain STS temporary credentials |
| **8** | Use STS credentials to perform **S3 bucket operations** |
| **9** | Clean up all created resources |

> **Prerequisites:** The Ceph cluster must be running (`docker compose up -d --wait`).  
> The notebook talks to RGW at the IP on `ceph-net`; adjust `RGW_ENDPOINT` below if you have
> host ports published.

In [7]:
# Install required packages
%pip install boto3 PyJWT cryptography requests -q
print("✓ Packages ready")

Note: you may need to restart the kernel to use updated packages.
✓ Packages ready


## Section 1 — Configuration and Admin Client

Edit `RGW_ENDPOINT` and `JWKS_HOST` to match your environment.
- **`RGW_ENDPOINT`** — RGW container IP on `ceph-net` (or `http://localhost:7480` if ports are published)
- **`JWKS_HOST`** — IP reachable *from inside the RGW container* to the notebook host.  
  Docker's gateway on `ceph-net` (`172.20.0.1`) works when the notebook runs on the same host.

In [31]:
import subprocess
import boto3
import json
import time
import threading
import base64
import requests
from datetime import datetime, timezone
from http.server import HTTPServer, BaseHTTPRequestHandler
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend
import jwt as pyjwt

# ── RGW / cluster ────────────────────────────────────────────────────────────
RGW_ENDPOINT = "http://172.20.0.13:7480"   # ceph-net IP; or http://localhost:7480
REGION       = "us-east-1"
COMPOSE_DIR  = "/home/harry/projects/ceph-lab"  # adjust if running notebook elsewhere

# ── Platform admin credentials (from setup.sh / .env) ────────────────────────
ADMIN_ACCESS_KEY = "iamaccess123456"
ADMIN_SECRET_KEY = "iamsecret1234567"

# ── Tenant namespace ──────────────────────────────────────────────────────────
TENANT       = "acmecorp"
TENANT_ADMIN = f"{TENANT}-admin"
TENANT_USER  = f"{TENANT}-user1"
TENANT_ROLE  = f"{TENANT}-s3role"
TENANT_BUCKET= f"{TENANT}-demo-bucket"

# ── OIDC / JWKS mini-server (runs on the notebook host) ──────────────────────
JWKS_HOST = "172.20.0.1"   # Docker gateway — reachable from inside RGW container
JWKS_PORT = 8877            # Must not conflict with other services

# ── RGW Admin API helper (via docker compose exec radosgw-admin) ──────────────
# NOTE: Ceph RGW Reef does not support IAM CreateUser/GetUser/ListUsers.
#       User management uses the radosgw-admin Admin API instead.
def rgw_admin(*args):
    """Run radosgw-admin inside the mon container and return parsed JSON."""
    cmd = ["docker", "compose", "exec", "-T", "mon", "radosgw-admin"] + list(args)
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=COMPOSE_DIR)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return json.loads(result.stdout) if result.stdout.strip() else {}

# ── boto3 client factories ────────────────────────────────────────────────────
def make_iam(access, secret):
    return boto3.client("iam", endpoint_url=RGW_ENDPOINT, region_name=REGION,
                        aws_access_key_id=access, aws_secret_access_key=secret)

def make_sts(access, secret):
    return boto3.client("sts", endpoint_url=RGW_ENDPOINT, region_name=REGION,
                        aws_access_key_id=access, aws_secret_access_key=secret)

def make_s3(access, secret, token=None):
    return boto3.client("s3", endpoint_url=RGW_ENDPOINT, region_name=REGION,
                        aws_access_key_id=access, aws_secret_access_key=secret,
                        aws_session_token=token)

# ── Pretty-printer ────────────────────────────────────────────────────────────
def pp(label, obj):
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print('─'*60)
    print(json.dumps(obj, indent=2, default=str))

# Platform admin IAM client (for roles / OIDC)
iam_admin = make_iam(ADMIN_ACCESS_KEY, ADMIN_SECRET_KEY)

print("✓ Configuration loaded")
print(f"  RGW endpoint : {RGW_ENDPOINT}")
print(f"  Tenant       : {TENANT}")
print(f"  JWKS server  : http://{JWKS_HOST}:{JWKS_PORT}")

✓ Configuration loaded
  RGW endpoint : http://172.20.0.13:7480
  Tenant       : acmecorp
  JWKS server  : http://172.20.0.1:8877


## Section 2 — Create Tenant and Tenant Administrator

The **platform admin** (`iamadmin`) creates:
1. A **Tenant Administrator** IAM user (`acmecorp-admin`)
2. An inline policy granting the Tenant Admin full IAM + STS rights
3. An access-key pair for the Tenant Admin so it can authenticate independently

> In Ceph RGW, tenancy is modelled through a shared user namespace.  
> Naming convention (`acmecorp-*`) keeps all tenant resources identifiable.

In [32]:
# ── 2a. Create Tenant Administrator user via Admin API ───────────────────────
# Note: RGW Reef IAM API does not support CreateUser; use radosgw-admin instead.
try:
    user_info = rgw_admin("user", "create",
                          f"--uid={TENANT_ADMIN}",
                          f"--display-name={TENANT} Administrator",
                          f"--email={TENANT_ADMIN}@{TENANT}.example.com")
    print(f"✓ Created user  : {user_info['user_id']}")
except RuntimeError as e:
    if "exists" in str(e).lower() or "EEXIST" in str(e) or "duplicate" in str(e).lower():
        print(f"ℹ  User '{TENANT_ADMIN}' already exists — fetching info")
        user_info = rgw_admin("user", "info", f"--uid={TENANT_ADMIN}")
    else:
        raise

# ── 2b. Grant Tenant Admin elevated RGW capabilities ─────────────────────────
#  roles=*         → create / delete / list IAM roles
#  users=*         → manage RGW users (for subaccount creation)
#  buckets=*       → inspect tenant buckets
#  oidc-provider=* → register OIDC providers
user_info = rgw_admin("caps", "add",
                      f"--uid={TENANT_ADMIN}",
                      "--caps=roles=*;users=*;buckets=*;oidc-provider=*;metadata=*")
caps = [f"{c['type']}={c['perm']}" for c in user_info['caps']]
print(f"✓ Capabilities  : {', '.join(caps)}")

# ── 2c. Extract (or create) access/secret key pair ───────────────────────────
keys = user_info.get("keys", [])
if not keys:
    user_info = rgw_admin("key", "create", f"--uid={TENANT_ADMIN}", "--key-type=s3")
    keys = user_info["keys"]

TENANT_ADMIN_ACCESS = keys[0]["access_key"]
TENANT_ADMIN_SECRET = keys[0]["secret_key"]

pp("Tenant Admin user info", {
    "user_id":    user_info["user_id"],
    "caps":       caps,
    "access_key": TENANT_ADMIN_ACCESS,
    "secret_key": TENANT_ADMIN_SECRET[:8] + "…",
})

# ── 2d. Build Tenant Admin IAM client ────────────────────────────────────────
iam_tenant = make_iam(TENANT_ADMIN_ACCESS, TENANT_ADMIN_SECRET)
print(f"\n✓ Tenant Admin IAM client ready for '{TENANT_ADMIN}'")

ℹ  User 'acmecorp-admin' already exists — fetching info
✓ Capabilities  : buckets=*, metadata=*, oidc-provider=*, roles=*, users=*

────────────────────────────────────────────────────────────
  Tenant Admin user info
────────────────────────────────────────────────────────────
{
  "user_id": "acmecorp-admin",
  "caps": [
    "buckets=*",
    "metadata=*",
    "oidc-provider=*",
    "roles=*",
    "users=*"
  ],
  "access_key": "PS2Z8WWKEHLT0XFEE77W",
  "secret_key": "W6W0CtTg\u2026"
}

✓ Tenant Admin IAM client ready for 'acmecorp-admin'


/home/harry/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


## Section 3 — Tenant Administrator Creates Subaccounts

Acting as the **Tenant Admin**, we create a subaccount user (`acmecorp-user1`).  
The subaccount is granted only a minimal STS policy — it will obtain S3 access  
exclusively through **role assumption**, not through direct S3 credentials.

In [33]:
# ── 3a. Create subaccount user via Admin API (as Tenant Admin) ───────────────
# Ceph RGW does not support IAM CreateUser; use radosgw-admin.
# The Tenant Admin's elevated caps (users=*) allow managing other users.
try:
    sub_info = rgw_admin("user", "create",
                         f"--uid={TENANT_USER}",
                         f"--display-name={TENANT} Subaccount User 1",
                         f"--email={TENANT_USER}@{TENANT}.example.com")
    print(f"✓ Created subaccount : {sub_info['user_id']}")
except RuntimeError as e:
    if "exists" in str(e).lower() or "EEXIST" in str(e) or "duplicate" in str(e).lower():
        print(f"ℹ  User '{TENANT_USER}' already exists — fetching info")
        sub_info = rgw_admin("user", "info", f"--uid={TENANT_USER}")
    else:
        raise

# ── 3b. Grant subaccount minimal caps: only STS AssumeRole ───────────────────
sub_info = rgw_admin("caps", "add",
                     f"--uid={TENANT_USER}",
                     "--caps=roles=read")   # read-only roles visibility for STS
sub_caps = [f"{c['type']}={c['perm']}" for c in sub_info['caps']]
print(f"✓ Subaccount caps    : {', '.join(sub_caps)}")

# ── 3c. Extract (or create) access/secret key pair ───────────────────────────
keys = sub_info.get("keys", [])
if not keys:
    sub_info = rgw_admin("key", "create", f"--uid={TENANT_USER}", "--key-type=s3")
    keys = sub_info["keys"]

SUBUSER_ACCESS = keys[0]["access_key"]
SUBUSER_SECRET = keys[0]["secret_key"]

pp("Subaccount user info", {
    "user_id":    sub_info["user_id"],
    "caps":       sub_caps,
    "access_key": SUBUSER_ACCESS,
    "secret_key": SUBUSER_SECRET[:8] + "…",
})

# ── 3d. List all users visible via radosgw-admin ─────────────────────────────
users = rgw_admin("user", "list", "--max-entries=100")
tenant_users = [u for u in users if TENANT in u]
print(f"\n  RGW users matching '{TENANT}' ({len(tenant_users)}):")
for u in tenant_users:
    print(f"    - {u}")

# ── 3e. Confirm roles visible to Tenant Admin IAM client ─────────────────────
roles = iam_tenant.list_roles()["Roles"]
print(f"\n  IAM roles visible to Tenant Admin ({len(roles)}):")
for r in roles:
    print(f"    - {r['RoleName']}  ({r['Arn']})")

ℹ  User 'acmecorp-user1' already exists — fetching info
✓ Subaccount caps    : roles=read

────────────────────────────────────────────────────────────
  Subaccount user info
────────────────────────────────────────────────────────────
{
  "user_id": "acmecorp-user1",
  "caps": [
    "roles=read"
  ],
  "access_key": "P56J69VAKIEKENJV2XJM",
  "secret_key": "SW7QAMpS\u2026"
}

  RGW users matching 'acmecorp' (0):

  IAM roles visible to Tenant Admin (2):
    - acmecorp-s3role  (arn:aws:iam:::role/acmecorp-s3role)
    - demo-role  (arn:aws:iam:::role/demo-role)


## Section 4 — Tenant-Specific IAM Role with Policy Documents

The **Tenant Admin** creates a role that:
- **Trust policy** — allows OIDC-authenticated identities (WebIdentity) whose `sub` claim
  matches the subaccount user to assume the role
- **Permission policy** — grants scoped S3 read/write access, restricted to buckets
  whose name starts with the tenant prefix (`acmecorp-*`)

The role ARN is captured for use in the STS call.

In [44]:
from botocore.exceptions import ClientError

# ── 4a. Trust policy — OIDC WebIdentity ──────────────────────────────────────
#   Principal is the OIDC provider ARN; Condition locks down to this subaccount user.
TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Federated": f"arn:aws:iam:::oidc-provider/{OIDC_PRINCIPAL}"
            },
            "Action": "sts:AssumeRoleWithWebIdentity",
            "Condition": {
                "StringEquals": {
                    f"{OIDC_PRINCIPAL}:sub": TENANT_USER,
                    f"{OIDC_PRINCIPAL}:aud": TENANT_USER,
                }
            }
        }
    ]
})

# ── 4b. Permission policy — scoped S3 access ─────────────────────────────────
# NOTE: s3:ListAllMyBuckets is a global (non-resource-scoped) action — it needs
# Resource: ["*"].  All object/bucket actions stay restricted to acmecorp-*.
S3_PERMISSION_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "ListAllBuckets",
            "Effect": "Allow",
            "Action": ["s3:ListAllMyBuckets"],
            "Resource": ["*"]
        },
        {
            "Sid": "TenantBucketAccess",
            "Effect": "Allow",
            "Action": [
                "s3:CreateBucket",
                "s3:DeleteBucket",
                "s3:ListBucket",
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject",
            ],
            "Resource": [
                f"arn:aws:s3:::{TENANT}-*",
                f"arn:aws:s3:::{TENANT}-*/*",
            ]
        }
    ]
})

pp("Trust Policy", json.loads(TRUST_POLICY))
pp("Permission Policy", json.loads(S3_PERMISSION_POLICY))

# ── 4c. Create the role ───────────────────────────────────────────────────────
# RGW Reef returns error code "Unknown" instead of EntityAlreadyExists when a
# role name is already taken — catch generic ClientError and fall back to listing.
try:
    resp = iam_tenant.create_role(
        RoleName=TENANT_ROLE,
        AssumeRolePolicyDocument=TRUST_POLICY,
        Description=f"Scoped S3 role for {TENANT} OIDC-authenticated users",
    )
    ROLE_ARN = resp["Role"]["Arn"]
    print(f"\n✓ Role created: {ROLE_ARN}")
except ClientError:
    roles = iam_tenant.list_roles()["Roles"]
    match = next((r for r in roles if r["RoleName"] == TENANT_ROLE), None)
    if match:
        ROLE_ARN = match["Arn"]
        print(f"ℹ  Role '{TENANT_ROLE}' already exists: {ROLE_ARN}")
    else:
        raise

# ── 4d. Attach permission policy to role ───────────────────────────────────────
# put_role_policy is idempotent in standard IAM; RGW may still error — tolerate it.
try:
    iam_tenant.put_role_policy(
        RoleName=TENANT_ROLE,
        PolicyName=f"{TENANT_ROLE}-s3-permissions",
        PolicyDocument=S3_PERMISSION_POLICY,
    )
    print(f"✓ S3 permission policy attached to role '{TENANT_ROLE}'")
except ClientError as e:
    print(f"ℹ  put_role_policy: {e}")

print(f"\n  Role ARN: {ROLE_ARN}")



────────────────────────────────────────────────────────────
  Trust Policy
────────────────────────────────────────────────────────────
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Federated": "arn:aws:iam:::oidc-provider/172.20.0.1:8877"
      },
      "Action": "sts:AssumeRoleWithWebIdentity",
      "Condition": {
        "StringEquals": {
          "172.20.0.1:8877:sub": "acmecorp-user1",
          "172.20.0.1:8877:aud": "acmecorp-user1"
        }
      }
    }
  ]
}

────────────────────────────────────────────────────────────
  Permission Policy
────────────────────────────────────────────────────────────
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "ListAllBuckets",
      "Effect": "Allow",
      "Action": [
        "s3:ListAllMyBuckets"
      ],
      "Resource": [
        "*"
      ]
    },
    {
      "Sid": "TenantBucketAccess",
      "Effect": "Allow",
      "Action": [
        "s3:CreateBuck

## Section 5 — OIDC Provider: Key Generation, JWKS Server, and Registration

To issue real OIDC tokens, this notebook runs a **self-contained JWKS/OIDC discovery server**
in a background thread:

```
http://<JWKS_HOST>:<JWKS_PORT>/.well-known/openid-configuration  ← discovery doc
http://<JWKS_HOST>:<JWKS_PORT>/.well-known/jwks.json             ← public key set
```

Steps:
1. Generate an **RSA-2048 key pair** in memory
2. Expose the public key as a **JSON Web Key Set (JWKS)**
3. Start the HTTP server on `0.0.0.0:JWKS_PORT` (reachable from RGW via `JWKS_HOST`)
4. Register the provider URL with Ceph RGW via `CreateOpenIDConnectProvider`

In [45]:
# ── 5a. Generate RSA-2048 key pair + self-signed X.509 certificate ───────────
# NOTE: Ceph RGW Reef requires the JWT header to contain an x5c (X.509
#       certificate chain). We generate a self-signed cert from the same key.
from cryptography import x509
from cryptography.x509.oid import NameOID
from datetime import datetime, timezone, timedelta   # re-bind name as class (not module)

_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend(),
)
_public_key  = _private_key.public_key()
_pub_numbers = _public_key.public_numbers()

PRIVATE_KEY_PEM = _private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=serialization.NoEncryption(),
)

# Self-signed certificate (public key = same RSA key)
_subject = _issuer = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "OIDC Lab")])
_now = datetime.now(timezone.utc)
_cert = (
    x509.CertificateBuilder()
    .subject_name(_subject)
    .issuer_name(_issuer)
    .public_key(_public_key)
    .serial_number(x509.random_serial_number())
    .not_valid_before(_now)
    .not_valid_after(_now + timedelta(days=365))
    .sign(_private_key, __import__("cryptography.hazmat.primitives.hashes", fromlist=["SHA256"]).SHA256())
)
CERT_DER  = _cert.public_bytes(serialization.Encoding.DER)
X5C_B64   = base64.b64encode(CERT_DER).decode()   # standard base64 (not url-safe)

KID = "lab-oidc-key-1"

def _int_to_b64url(n: int) -> str:
    length = (n.bit_length() + 7) // 8
    return base64.urlsafe_b64encode(n.to_bytes(length, "big")).rstrip(b"=").decode()

JWK_SET = {
    "keys": [
        {
            "kty": "RSA",
            "use": "sig",
            "alg": "RS256",
            "kid": KID,
            "n":   _int_to_b64url(_pub_numbers.n),
            "e":   _int_to_b64url(_pub_numbers.e),
            "x5c": [X5C_B64],   # cert chain — required by Ceph RGW Reef
        }
    ]
}

OIDC_DISCOVERY = {
    "issuer":                                OIDC_ISSUER,
    "jwks_uri":                              f"{OIDC_ISSUER}/.well-known/jwks.json",
    "response_types_supported":              ["id_token"],
    "subject_types_supported":               ["public"],
    "id_token_signing_alg_values_supported": ["RS256"],
}

print("✓ RSA-2048 key pair + self-signed X.509 cert generated")
print(f"  KID     : {KID}")
print(f"  Cert CN : OIDC Lab")
print(f"  x5c     : {X5C_B64[:40]}…")


✓ RSA-2048 key pair + self-signed X.509 cert generated
  KID     : lab-oidc-key-1
  Cert CN : OIDC Lab
  x5c     : MIICsjCCAZqgAwIBAgIUaat72H5TRlTD7udcnWmt…


In [ ]:
# ── 5b. Start OIDC/JWKS HTTP server in background thread ─────────────────────
import socket

_jwks_json      = json.dumps(JWK_SET).encode()
_discovery_json = json.dumps(OIDC_DISCOVERY).encode()

class _OIDCHandler(BaseHTTPRequestHandler):
    _routes = {
        "/.well-known/openid-configuration": _discovery_json,
        "/.well-known/jwks.json":            _jwks_json,
    }

    def do_GET(self):
        body = self._routes.get(self.path)
        if body is None:
            self.send_response(404); self.end_headers(); return
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, fmt, *args):
        print(f"  [OIDC server] {self.address_string()} {fmt % args}")

# Stop any previously running server (safe on re-run)
try:
    _oidc_server.shutdown()   # stops serve_forever loop
    _oidc_server.server_close()  # closes the listening socket
    time.sleep(0.3)
    print("ℹ  Previous OIDC server stopped")
except NameError:
    pass

_oidc_server = HTTPServer(("0.0.0.0", JWKS_PORT), _OIDCHandler)
_oidc_server.allow_reuse_address = True
threading.Thread(target=_oidc_server.serve_forever, daemon=True).start()
print(f"✓ OIDC server listening on 0.0.0.0:{JWKS_PORT}")

# Self-test from notebook host
r = requests.get(f"http://localhost:{JWKS_PORT}/.well-known/openid-configuration", timeout=5)
r.raise_for_status()
print(f"✓ Discovery endpoint OK — issuer: {r.json()['issuer']}")

r = requests.get(f"http://localhost:{JWKS_PORT}/.well-known/jwks.json", timeout=5)
r.raise_for_status()
jwks = r.json()
print(f"✓ JWKS endpoint OK — {len(jwks['keys'])} key(s), x5c present: {'x5c' in jwks['keys'][0]}")

ℹ  Previous OIDC server stopped
✓ OIDC server listening on 0.0.0.0:8877
  [OIDC server] 127.0.0.1 "GET /.well-known/openid-configuration HTTP/1.1" 200 -
✓ Discovery endpoint OK — issuer: http://172.20.0.1:8877
  [OIDC server] 127.0.0.1 "GET /.well-known/jwks.json HTTP/1.1" 200 -
✓ JWKS endpoint OK — 1 key(s), x5c present: True


  [OIDC server] 172.20.0.13 "GET /.well-known/openid-configuration HTTP/1.1" 200 -
  [OIDC server] 172.20.0.13 "GET /.well-known/jwks.json HTTP/1.1" 200 -


In [47]:
# ── 5c. Register OIDC provider with Ceph RGW ─────────────────────────────────
#
# ThumbprintList is required by the API but not validated by RGW for HTTP endpoints.
# A 40-character dummy is used here; in production use the real TLS cert SHA-1 fingerprint.
#
# RGW Reef returns "Unknown" instead of EntityAlreadyExists — catch ClientError.
try:
    resp = iam_admin.create_open_id_connect_provider(
        Url=OIDC_ISSUER,
        ClientIDList=[TENANT_USER],          # audiences this provider issues tokens for
        ThumbprintList=["0" * 40],           # dummy — not checked for HTTP in Ceph
    )
    OIDC_ARN = resp["OpenIDConnectProviderArn"]
    print(f"✓ OIDC provider registered")
except ClientError:
    OIDC_ARN = f"arn:aws:iam:::oidc-provider/{OIDC_PRINCIPAL}"
    print(f"ℹ  OIDC provider already registered")

print(f"  ARN: {OIDC_ARN}")


ℹ  OIDC provider already registered
  ARN: arn:aws:iam:::oidc-provider/172.20.0.1:8877


In [48]:
# ── 5d. Generate signed OIDC ID Token (JWT) with x5c header ──────────────────
# Ceph RGW Reef requires the x5c JWT header; it uses the certificate's public
# key to verify the token signature (in addition to or instead of JWKS kid lookup).
now = int(time.time())

id_token_claims = {
    "iss": OIDC_ISSUER,                # must match registered provider URL
    "sub": TENANT_USER,                # subject — must match role trust policy condition
    "aud": TENANT_USER,                # audience — must match ClientIDList above
    "iat": now,
    "exp": now + 3600,
    # Custom claims (informational)
    "name":     f"{TENANT} Subaccount User",
    "tenant":   TENANT,
    "email":    f"{TENANT_USER}@{TENANT}.example.com",
}

_encoded = pyjwt.encode(
    id_token_claims,
    PRIVATE_KEY_PEM,
    algorithm="RS256",
    headers={"kid": KID, "x5c": [X5C_B64]},   # x5c required by Ceph RGW Reef
)
# PyJWT 1.x returns bytes; 2.x returns str — normalise to str
ID_TOKEN = _encoded.decode("utf-8") if isinstance(_encoded, bytes) else _encoded

# Display decoded claims (disable sig + aud verification — for inspection only)
decoded = pyjwt.decode(
    ID_TOKEN,
    options={"verify_signature": False, "verify_aud": False},
)
pp("ID Token Claims", decoded)

import json as _json
header = _json.loads(base64.urlsafe_b64decode(ID_TOKEN.split(".")[0] + "=="))
print(f"\n✓ JWT Header    : {header}")
print(f"  x5c present  : {'x5c' in header}")
print(f"  Token        : {ID_TOKEN[:60]}…")


────────────────────────────────────────────────────────────
  ID Token Claims
────────────────────────────────────────────────────────────
{
  "iss": "http://172.20.0.1:8877",
  "sub": "acmecorp-user1",
  "aud": "acmecorp-user1",
  "iat": 1773356779,
  "exp": 1773360379,
  "name": "acmecorp Subaccount User",
  "tenant": "acmecorp",
  "email": "acmecorp-user1@acmecorp.example.com"
}

✓ JWT Header    : {'typ': 'JWT', 'alg': 'RS256', 'kid': 'lab-oidc-key-1', 'x5c': ['MIICsjCCAZqgAwIBAgIUaat72H5TRlTD7udcnWmtRS6bY0cwDQYJKoZIhvcNAQELBQAwEzERMA8GA1UEAwwIT0lEQyBMYWIwHhcNMjYwMzEyMjMwNjA5WhcNMjcwMzEyMjMwNjA5WjATMREwDwYDVQQDDAhPSURDIExhYjCCASIwDQYJKoZIhvcNAQEBBQADggEPADCCAQoCggEBANb5hozDCZ0DBgRa1a69ts9fsJYViCQZPzMu7j0ppW1R9LSiCXfFd1RXFsOb57rPwrozh7T/+ueuaxJm2ylniI3tOnA4TKut0S+i1/bTAOA8V8ATJ6l3188cK1lCEfgJwCek3h36Au6JmN9USZsf1PExoJioT5JMO4JxrlsUKo47hEmbzYURnrHnen8ip5KCpxiOIEJbdlVlYl9WIiL+xh1z8vCciKZXw1q39DODb6bUxV6thBxWTfAsK1CTE4sXuOk9D4vC0mvu40YrSLj/s9zhzA4MjNjDG+orlf2WKX2Jzmj5gj98HIae5nSNnb9vt

## Section 6 — STS: AssumeRoleWithWebIdentity

The subaccount user now calls **`AssumeRoleWithWebIdentity`** passing:
- **`RoleArn`** — the tenant role ARN created in Section 4
- **`WebIdentityToken`** — the RS256-signed JWT from Section 5
- **`RoleSessionName`** — an identifying label for this session

RGW validates the JWT:
1. Fetches `<issuer>/.well-known/openid-configuration` → finds `jwks_uri`
2. Fetches `jwks_uri` → finds the public key matching `kid`
3. Verifies the JWT signature and checks `sub` / `aud` conditions in the trust policy
4. Issues temporary credentials (AccessKeyId, SecretAccessKey, SessionToken)

In [49]:
# ── 6. AssumeRoleWithWebIdentity ─────────────────────────────────────────────
sts_client = make_sts(SUBUSER_ACCESS, SUBUSER_SECRET)

resp = sts_client.assume_role_with_web_identity(
    RoleArn=ROLE_ARN,
    RoleSessionName=f"{TENANT_USER}-oidc-session",
    WebIdentityToken=ID_TOKEN,
    DurationSeconds=3600,
)

creds       = resp["Credentials"]
STS_ACCESS  = creds["AccessKeyId"]
STS_SECRET  = creds["SecretAccessKey"]
STS_TOKEN   = creds["SessionToken"]
STS_EXPIRY  = creds["Expiration"]

assumed_user = resp.get("AssumedRoleUser", {})

pp("AssumeRoleWithWebIdentity response", {
    "AssumedRoleUser": assumed_user,
    "Credentials": {
        "AccessKeyId":    STS_ACCESS,
        "SecretKey":      STS_SECRET[:8] + "…",
        "SessionToken":   STS_TOKEN[:20] + "…",
        "Expiration":     STS_EXPIRY,
    },
    "Provider": resp.get("Provider"),
    "Audience":  resp.get("Audience"),
})

print(f"\n✓ Temporary credentials obtained!")
print(f"  Expires: {STS_EXPIRY}")


────────────────────────────────────────────────────────────
  AssumeRoleWithWebIdentity response
────────────────────────────────────────────────────────────
{
  "AssumedRoleUser": {
    "Arn": "arn:aws:sts:::assumed-role/acmecorp-s3role/acmecorp-user1-oidc-session"
  },
  "Credentials": {
    "AccessKeyId": "fzBtAeQyvU0CkD3N3Zzb",
    "SecretKey": "Z7O7VWDS\u2026",
    "SessionToken": "EQUtb/9C3g1csFPOLF/S\u2026",
    "Expiration": "2026-03-13 00:06:23.365478+00:00"
  },
  "Provider": "http://172.20.0.1:8877",
  "Audience": "acmecorp-user1"
}

✓ Temporary credentials obtained!
  Expires: 2026-03-13 00:06:23.365478+00:00


## Section 7 — S3 Bucket Operations Using STS Credentials

The **temporary credentials** from the STS call are now used to drive the full S3 lifecycle:

| Operation | boto3 call |
|---|---|
| Create bucket | `create_bucket` |
| List buckets | `list_buckets` |
| Upload object | `put_object` |
| List objects | `list_objects_v2` |
| Download and verify | `get_object` |
| Delete object | `delete_object` |
| Delete bucket | `delete_bucket` |

The bucket name starts with `acmecorp-`, so it falls within the permission policy's `Resource` scope.

In [50]:
# ── 7. S3 operations with STS credentials ────────────────────────────────────
s3 = make_s3(STS_ACCESS, STS_SECRET, STS_TOKEN)

OBJECT_KEY  = "demo/hello.txt"
OBJECT_BODY = (
    f"Written by : {TENANT_USER}\n"
    f"Role       : {TENANT_ROLE}\n"
    f"Timestamp  : {datetime.now(timezone.utc).isoformat()}\n"
    f"Auth       : OIDC → STS AssumeRoleWithWebIdentity\n"
)

# ── 7a. Create bucket ─────────────────────────────────────────────────────────
try:
    s3.create_bucket(Bucket=TENANT_BUCKET)
    print(f"✓ Bucket created : s3://{TENANT_BUCKET}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"ℹ  Bucket already exists : s3://{TENANT_BUCKET}")
    else:
        raise

# ── 7b. List objects in bucket (confirms bucket accessible + is empty) ────────
init_objs = s3.list_objects_v2(Bucket=TENANT_BUCKET).get("Contents", [])
print(f"\n✓ Bucket accessible : s3://{TENANT_BUCKET}  ({len(init_objs)} objects)")

# ── 7c. Upload object ─────────────────────────────────────────────────────────
s3.put_object(Bucket=TENANT_BUCKET, Key=OBJECT_KEY, Body=OBJECT_BODY.encode())
print(f"\n✓ Uploaded  : s3://{TENANT_BUCKET}/{OBJECT_KEY}")

# ── 7d. List objects ──────────────────────────────────────────────────────────
objs = s3.list_objects_v2(Bucket=TENANT_BUCKET).get("Contents", [])
print(f"\n✓ Objects in bucket ({len(objs)}):")
for o in objs:
    print(f"    {o['Key']}  ({o['Size']} bytes,  ETag: {o['ETag']})")

# ── 7e. Download and verify ───────────────────────────────────────────────────
downloaded = s3.get_object(Bucket=TENANT_BUCKET, Key=OBJECT_KEY)["Body"].read().decode()
print(f"\n✓ Downloaded object content:")
print("  " + downloaded.replace("\n", "\n  ").rstrip())
assert downloaded == OBJECT_BODY, "Content mismatch!"
print("  ✓ Round-trip content matches")


✓ Bucket created : s3://acmecorp-demo-bucket

✓ Bucket accessible : s3://acmecorp-demo-bucket  (0 objects)

✓ Uploaded  : s3://acmecorp-demo-bucket/demo/hello.txt

✓ Objects in bucket (1):
    demo/hello.txt  (155 bytes,  ETag: "af5d20f5a723be52a0dad3640c4ebe0e")

✓ Downloaded object content:
  Written by : acmecorp-user1
  Role       : acmecorp-s3role
  Timestamp  : 2026-03-12T23:06:26.344114+00:00
  Auth       : OIDC → STS AssumeRoleWithWebIdentity
  ✓ Round-trip content matches


## Section 8 — Cleanup

Remove all resources created during this notebook run:
- S3 objects and bucket
- IAM role and its inline policy
- Subaccount user, access key, inline policy
- Tenant Admin user, access key, inline policy
- OIDC provider registration
- Stop the background JWKS server

In [51]:
def _safe(label, fn):
    try:
        fn()
        print(f"  ✓ {label}")
    except Exception as exc:
        print(f"  ⚠ {label}: {exc}")

def _safe_rgw(label, *args):
    _safe(label, lambda: rgw_admin(*args))

print("── S3 cleanup ───────────────────────────────────────────────────────")
_safe("delete object",  lambda: s3.delete_object(Bucket=TENANT_BUCKET, Key=OBJECT_KEY))
_safe("delete bucket",  lambda: s3.delete_bucket(Bucket=TENANT_BUCKET))

print("\n── IAM role cleanup (as Tenant Admin) ───────────────────────────────")
_safe("delete role inline policy",
      lambda: iam_tenant.delete_role_policy(
          RoleName=TENANT_ROLE, PolicyName=f"{TENANT_ROLE}-s3-permissions"))
_safe("delete role",
      lambda: iam_tenant.delete_role(RoleName=TENANT_ROLE))

print("\n── OIDC provider cleanup ────────────────────────────────────────────")
_safe("delete OIDC provider",
      lambda: iam_admin.delete_open_id_connect_provider(
          OpenIDConnectProviderArn=OIDC_ARN))

print("\n── Subaccount user cleanup (via Admin API) ──────────────────────────")
_safe_rgw("delete subaccount user", "user", "rm",
          f"--uid={TENANT_USER}", "--purge-data")

print("\n── Tenant Admin user cleanup (via Admin API) ────────────────────────")
_safe_rgw("delete tenant-admin user", "user", "rm",
          f"--uid={TENANT_ADMIN}", "--purge-data")

print("\n── JWKS server shutdown ─────────────────────────────────────────────")
_safe("stop JWKS server", lambda: _oidc_server.shutdown())

print("\n✓ Cleanup complete")

── S3 cleanup ───────────────────────────────────────────────────────
  ✓ delete object
  ✓ delete bucket

── IAM role cleanup (as Tenant Admin) ───────────────────────────────
  ✓ delete role inline policy
  ✓ delete role

── OIDC provider cleanup ────────────────────────────────────────────
  ✓ delete OIDC provider

── Subaccount user cleanup (via Admin API) ──────────────────────────
  ✓ delete subaccount user

── Tenant Admin user cleanup (via Admin API) ────────────────────────
  ✓ delete tenant-admin user

── JWKS server shutdown ─────────────────────────────────────────────
  ✓ stop JWKS server

✓ Cleanup complete


## Summary

This notebook demonstrated the complete **IAM → OIDC → STS → S3** workflow on Ceph RGW:

```
Platform Admin (iamadmin)
  └─ creates → Tenant Admin (acmecorp-admin)
                  └─ creates → Subaccount (acmecorp-user1)
                  └─ creates → IAM Role (acmecorp-s3role)
                                  ├─ trust policy  : OIDC issuer + sub==acmecorp-user1
                                  └─ perm policy   : s3:* on arn:aws:s3:::acmecorp-*

OIDC Flow (self-signed, local JWKS server):
  Notebook JWKS server  ──JWKS──►  Ceph RGW validates JWT
  acmecorp-user1 (JWT)  ──AssumeRoleWithWebIdentity──►  STS temp creds

S3 (temp creds):
  create bucket → upload object → list → download → delete → delete bucket
```

### Key takeaways

| Topic | Detail |
|---|---|
| **Tenancy model** | Naming convention (`acmecorp-*`) scopes users, roles, and buckets |
| **OIDC validation** | RGW fetches JWKS from `<issuer>/.well-known/jwks.json` and verifies RS256 JWT |
| **Credential lifetime** | STS tokens expire (here: 1 hour); refresh by calling `AssumeRoleWithWebIdentity` again |
| **Policy scoping** | Permission policy restricts S3 access to `arn:aws:s3:::acmecorp-*` |
| **`rgw_sts_key`** | Must be exactly 16 bytes; pass via `--rgw-sts-key` CLI flag, **not** via config DB |